# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and practical example for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities (record sets, fields, columns) by their `@id` in accordance with FAIR principles.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Load and pretty-print metadata
meta = dataset.metadata
print(f"Dataset title: {meta.name}\n{meta.description}\n\nCite as: {getattr(meta, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` attributes. All references will use `@id` notation as per FAIR principles.

In [ ]:
# Inspect the available record sets and their fields by `@id`.
record_sets = list(dataset.record_sets)
print(f"Number of record sets in dataset: {len(record_sets)}")

all_recordset_info = []
for rec in record_sets:
    info = {
        '@id': rec.id_,
        'name': rec.name,
        'description': getattr(rec, 'description', ''),
        'fields': [(f.id_, f.name) for f in rec.fields],
    }
    all_recordset_info.append(info)
    print(f"\nRecordSet @id: {rec.id_}\n  Name: {rec.name}\n  Description: {getattr(rec, 'description', '')}")
    print("  Fields and their @id:")
    for field in rec.fields:
        print(f"    - {field.id_}: {field.name}")

# For demonstration, pick the first record set's @id
example_record_set_id = record_sets[0].id_

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the desired record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for rec in record_sets:
    df = pd.DataFrame(list(dataset.records(record_set=rec.id_)))
    dataframes[rec.id_] = df

# Display columns of the example record set
df = dataframes[example_record_set_id]
print(f"Columns for record set {example_record_set_id}:\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
The following demonstrates typical data processing steps, referencing fields by their Croissant `@id`. Adjust the field `@id` selection for your downstream analysis.

In [ ]:
import numpy as np

# Choose a numeric field by its @id for analysis
numeric_field_id = None
group_field_id = None
# Loop to auto-detect the first numeric-typed field
for field in record_sets[0].fields:
    # Check if the field appears to be numeric by dtype in the DataFrame
    if field.id_ in df.columns and np.issubdtype(df[field.id_].dtype, np.number):
        numeric_field_id = field.id_
        break

# Choose a group field (categorical) if available
if not numeric_field_id:
    # Fallback: try columns containing typical numeric phrases (e.g., 'score')
    for col in df.columns:
        if 'abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
            break
if not numeric_field_id:
    # If still not found, use the first column
    numeric_field_id = df.columns[0]

# Try to pick a group field
for field in record_sets[0].fields:
    if field.id_ != numeric_field_id and df[field.id_].dtype == object:
        group_field_id = field.id_
        break
if not group_field_id:
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group/categorical field @id: {group_field_id}\n")
# Remove missing/invalid values for the selected numeric field
working_df = df.copy()
working_df = working_df[pd.to_numeric(working_df[numeric_field_id], errors='coerce').notnull()]
working_df[numeric_field_id] = pd.to_numeric(working_df[numeric_field_id], errors='coerce')

# Threshold for numeric filtering (median + 1 std for demo)
mean = working_df[numeric_field_id].mean()
std = working_df[numeric_field_id].std()
threshold = mean + std
filtered_df = working_df[working_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - working_df[numeric_field_id].mean()) / working_df[numeric_field_id].std()
print("\nNormalized numeric field:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the selected field, calculate mean
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped (mean) by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we use matplotlib and seaborn for demonstration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(working_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If group_field_id is reasonably categorical (not too many groups), plot boxplot
if working_df[group_field_id].nunique() < 20:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=working_df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated: loading Croissant metadata and records with `mlcroissant`, exploring record sets and fields by `@id`, extracting tables, filtering/normalizing/grouping records, and basic visualization. Use fields' `@id` throughout to ensure reproducibility and interoperability. For further exploration, refer to the [mlcroissant documentation](https://mlcommons.org/croissant/spec/) and the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).